# 01 · Build the checkout order table

**Question.** For an order placed today, will its delivery arrive after the promised calendar date? The unit of analysis is one completed Olist order. Actual arrival supplies the label and the time when it became observable. The model receives the 19 fields in the checkout feature list.

Run this notebook first. It reads the original Olist release and writes one local Parquet file used by the other notebooks. The audit tables below contain aggregate counts.

In [1]:
import hashlib
import io
import json
import shutil
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile, is_zipfile

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/checkout_features.json").is_file())
RAW = ROOT / "data/raw"
PROCESSED = ROOT / "data/processed"
TABLES = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
for directory in (RAW, PROCESSED, TABLES, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

## Read the versioned source

The [Olist data card](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) describes orders with several items and potentially several sellers. Seven of its nine tables supply the label, transaction, catalog and location fields used here. The source manifest pins each input CSV by its SHA-256 hash. Existing extracted CSVs or a local ZIP both work.

In [2]:
manifest = json.loads((ROOT / "data/source_files.json").read_text())
archive = RAW / "olist-v2.zip"
source_url = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce?datasetVersionNumber=2"

if not all((RAW / name).is_file() for name in manifest) and not archive.exists():
    temporary = archive.with_suffix(".download")
    try:
        with urlopen(source_url, timeout=90) as response, temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
        if not is_zipfile(temporary):
            raise ValueError("Expected the Olist ZIP. See data/README.md for manual download.")
        temporary.replace(archive)
    finally:
        temporary.unlink(missing_ok=True)

def read_source(name, columns, dtype=None):
    path = RAW / name
    if path.is_file():
        contents = path.read_bytes()
    else:
        with ZipFile(archive) as source:
            contents = source.read(name)
    if hashlib.sha256(contents).hexdigest() != manifest[name]:
        raise ValueError(f"Unexpected contents of {name}. Use the version in data/README.md.")
    return pd.read_csv(io.BytesIO(contents), usecols=columns, dtype=dtype, encoding="utf-8-sig")

orders = read_source("olist_orders_dataset.csv", [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_estimated_delivery_date", "order_delivered_customer_date",
])
items = read_source("olist_order_items_dataset.csv", [
    "order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value",
])
products = read_source("olist_products_dataset.csv", [
    "product_id", "product_category_name", "product_weight_g",
    "product_length_cm", "product_height_cm", "product_width_cm",
])
customers = read_source("olist_customers_dataset.csv", [
    "customer_id", "customer_zip_code_prefix", "customer_state",
], {"customer_zip_code_prefix": str})
sellers = read_source("olist_sellers_dataset.csv", [
    "seller_id", "seller_zip_code_prefix", "seller_state",
], {"seller_zip_code_prefix": str})
geo = read_source("olist_geolocation_dataset.csv", [
    "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
], {"geolocation_zip_code_prefix": str})
translation = read_source("product_category_name_translation.csv", [
    "product_category_name", "product_category_name_english",
])

tables = {"orders": orders, "items": items, "products": products, "customers": customers,
          "sellers": sellers, "geolocation": geo, "category translations": translation}
display(pd.Series({k: len(v) for k, v in tables.items()}, name="source_rows").to_frame())
assert orders.order_id.is_unique and not items.duplicated(["order_id", "order_item_id"]).any()
assert all(table[key].notna().all() and table[key].is_unique for table, key in [
    (orders, "order_id"), (products, "product_id"), (customers, "customer_id"),
    (sellers, "seller_id"), (translation, "product_category_name"),
])

,source_rows
orders,99441
items,112650
products,32951
customers,99441
sellers,3095
geolocation,1000163
category translations,71


## Define an observable delivery outcome

Use delivered status and valid purchase, promise and arrival dates. The published timestamps have no timezone offset, so calendar dates use the source clock. An arrival on the promised day counts as on time. The timestamp comparison below measures how much a midnight deadline would change the target.

In [3]:
orders = orders.rename(columns={"order_purchase_timestamp": "purchased_at",
                                "order_delivered_customer_date": "label_observed_at",
                                "order_estimated_delivery_date": "promised_at"})
dates = ["purchased_at", "label_observed_at", "promised_at"]
for col in dates:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")
delivered = orders.loc[orders.order_status.eq("delivered")].copy()
dated = delivered.dropna(subset=dates)
sensible = (dated.label_observed_at >= dated.purchased_at) & (
    dated.promised_at.dt.normalize() >= dated.purchased_at.dt.normalize()
)
cohort = dated.loc[sensible].copy()
purchase_day = cohort.purchased_at.dt.normalize()
promise_day = cohort.promised_at.dt.normalize()
arrival_day = cohort.label_observed_at.dt.normalize()
cohort["late"] = (arrival_day > promise_day).astype("int8")
cohort["promise_days"] = (promise_day - purchase_day).dt.days

audit = pd.Series({
    "Source orders": len(orders), "Completed deliveries": len(delivered),
    "Complete required dates": len(dated), "Valid date chronology": len(cohort),
}, name="orders").to_frame()
display(audit)
audit.to_csv(TABLES / "cohort_audit.csv")
rules = pd.DataFrame({
    "promised calendar day": cohort.late,
    "exact promised timestamp": (cohort.label_observed_at > cohort.promised_at).astype(int),
}).agg(["sum", "mean"]).T.rename(columns={"sum": "late_orders", "mean": "late_rate"})
rules["late_orders"] = rules.late_orders.astype(int)
display(rules.style.format({"late_orders": "{:,}", "late_rate": "{:.2%}"}))
rules.to_csv(TABLES / "label_definitions.csv", index_label="rule")
assert (cohort.promised_at == promise_day).all()
assert (rules.loc["exact promised timestamp", "late_orders"] - cohort.late.sum()
        == ((arrival_day == promise_day) & (cohort.label_observed_at > promise_day)).sum())

,orders
Source orders,99441
Completed deliveries,96478
Complete required dates,96470
Valid date chronology,96470


,late_orders,late_rate
promised calendar day,"6,534",6.77%
exact promised timestamp,"7,826",8.11%


## Locate buyers and sellers

Each postal prefix maps to one median latitude and longitude. We remove coordinates outside a broad box around Brazil before calculating the medians. Catalog attributes and prefix coordinates serve as static checkout references. Distance is a straight-line approximation in kilometres.

In [4]:
def zip5(series):
    return series.astype("string").str.strip().str.zfill(5)

valid_geo = geo.geolocation_lat.between(-35, 6) & geo.geolocation_lng.between(-75, -28)
geo["prefix"] = zip5(geo.geolocation_zip_code_prefix)
centres = (geo.loc[valid_geo].groupby("prefix")[["geolocation_lat", "geolocation_lng"]]
           .median().rename(columns={"geolocation_lat": "lat", "geolocation_lng": "lon"}))

def locate(frame, prefix, role):
    return (frame.assign(prefix=zip5(frame[prefix]))
            .join(centres.add_prefix(role + "_"), on="prefix", validate="many_to_one")
            .drop(columns=[prefix, "prefix"]))

buyers = locate(customers, "customer_zip_code_prefix", "buyer").rename(columns={"customer_state": "buyer_state"})
vendors = locate(sellers, "seller_zip_code_prefix", "seller")
cohort = cohort.merge(buyers, on="customer_id", how="left", validate="many_to_one")
print(f"{len(centres):,} postal prefixes; {(~valid_geo).sum():,} coordinate rows outside the bounding box.")

19,011 postal prefixes; 29 coordinate rows outside the bounding box.


## Join item-level information

The product and seller joins preserve the composite item key. Weight is in kilograms and volume in litres. A nonpositive dimension or weight has unknown physical size. We keep these items and carry uncertainty into the order aggregate.

In [5]:
catalog = products.merge(translation, on="product_category_name", how="left", validate="many_to_one")
catalog["category"] = catalog.product_category_name_english.fillna(catalog.product_category_name)
shape_columns = ["product_length_cm", "product_height_cm", "product_width_cm"]
dimensions = catalog[shape_columns].where(catalog[shape_columns] > 0)
catalog["unit_volume_l"] = dimensions.prod(axis=1, min_count=3) / 1000
catalog["unit_weight_kg"] = catalog.product_weight_g.where(catalog.product_weight_g > 0) / 1000
eligible_items = items.loc[items.order_id.isin(cohort.order_id)]
lines = (eligible_items.merge(catalog[["product_id", "category", "unit_weight_kg", "unit_volume_l"]],
                              how="left", on="product_id", validate="many_to_one")
         .merge(vendors, on="seller_id", how="left", validate="many_to_one")
         .merge(cohort[["order_id", "buyer_state", "buyer_lat", "buyer_lon"]],
                on="order_id", how="left", validate="many_to_one"))
assert len(lines) == len(eligible_items)
assert not lines.duplicated(["order_id", "order_item_id"]).any()
for col in ["price", "freight_value"]:
    lines[col] = lines[col].where(lines[col] >= 0)

def haversine_km(lat_a, lon_a, lat_b, lon_b):
    lat_a, lon_a, lat_b, lon_b = map(np.deg2rad, (lat_a, lon_a, lat_b, lon_b))
    chord = (np.sin((lat_b - lat_a) / 2) ** 2
             + np.cos(lat_a) * np.cos(lat_b) * np.sin((lon_b - lon_a) / 2) ** 2)
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(chord, 0, 1)))

lines["route_km"] = haversine_km(lines.seller_lat, lines.seller_lon, lines.buyer_lat, lines.buyer_lon)
state_known = lines.buyer_state.notna() & lines.seller_state.notna()
lines["interstate"] = (lines.buyer_state != lines.seller_state).astype(float).where(state_known)

## Aggregate without multiplying orders

Freight and merchandise are totals across items. Use the longest seller-to-buyer distance for a multi-seller order. A total, a maximum or a share is missing when any contributing item value is missing. The highest-priced item defines the primary category and seller state, with the smallest item sequence breaking equal-price ties.

In [6]:
by_order = lines.groupby("order_id", sort=False)
basket = by_order.agg(
    item_count=("order_item_id", "size"), product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"), merchandise_brl=("price", "sum"),
    freight_brl=("freight_value", "sum"), max_item_price_brl=("price", "max"),
    weight_kg=("unit_weight_kg", "sum"), volume_l=("unit_volume_l", "sum"),
    distance_km=("route_km", "max"), interstate_item_share=("interstate", "mean"),
)
source_for_aggregate = {
    "merchandise_brl": "price", "max_item_price_brl": "price", "freight_brl": "freight_value",
    "weight_kg": "unit_weight_kg", "volume_l": "unit_volume_l", "distance_km": "route_km",
    "interstate_item_share": "interstate", "product_count": "product_id", "seller_count": "seller_id",
}
counts = by_order[list(dict.fromkeys(source_for_aggregate.values()))].count()
for aggregate, source in source_for_aggregate.items():
    basket.loc[counts[source] < basket.item_count, aggregate] = np.nan

primary = (lines.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True])
           .drop_duplicates("order_id").set_index("order_id")[["category", "seller_state"]]
           .rename(columns={"category": "primary_category", "seller_state": "primary_seller_state"}))
frame = cohort.join(basket.join(primary, validate="one_to_one"), on="order_id", validate="one_to_one")
assert len(frame) == len(cohort) and frame.order_id.is_unique
assert int(frame.item_count.sum()) == len(lines)
assert np.isclose(frame.merchandise_brl.sum(), lines.price.sum())  # complete prices in the pinned release
assert np.isclose(frame.freight_brl.sum(), lines.freight_value.sum())
audit_joins = pd.Series({
    "Eligible order items": len(eligible_items), "After item dimension joins": len(lines),
    "Orders with no matched items": frame.item_count.isna().sum(),
    "Orders with multiple sellers": frame.seller_count.gt(1).sum(),
    "Orders with incomplete route distance": frame.distance_km.isna().sum(),
    "Orders with incomplete package weight": frame.weight_kg.isna().sum(),
    "Orders with incomplete package volume": frame.volume_l.isna().sum(),
}, name="count").to_frame()
display(audit_joins)
audit_joins.to_csv(TABLES / "join_audit.csv")

,count
Eligible order items,110189
After item dimension joins,110189
Orders with no matched items,0
Orders with multiple sellers,1275
Orders with incomplete route distance,481
Orders with incomplete package weight,22
Orders with incomplete package volume,16


## Add purchase-time features and reserve later orders

Seasonality uses the source purchase clock. Cut at the calendar days containing the 70th and 85th percentiles of orders, placing each entire boundary day in the later partition. The remaining notebooks use these same assignments. Identifiers and the delivery timestamp stay in the local file for joining, tie-breaking and label-maturity checks.

In [7]:
month = frame.purchased_at.dt.month - 1
hour = frame.purchased_at.dt.hour + frame.purchased_at.dt.minute / 60
for name, phase, cycle in [("month", month, 12), ("hour", hour, 24)]:
    frame[f"purchase_{name}_sin"] = np.sin(2 * np.pi * phase / cycle)
    frame[f"purchase_{name}_cos"] = np.cos(2 * np.pi * phase / cycle)
frame["purchase_weekday"] = frame.purchased_at.dt.dayofweek.map(dict(enumerate(
    ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))
frame = frame.sort_values(["purchased_at", "order_id"]).reset_index(drop=True)
days = frame.purchased_at.dt.normalize()
val_start, test_start = [days.iloc[int(len(days) * fraction)] for fraction in (0.70, 0.85)]
frame["split"] = np.select([days < val_start, days < test_start], ["train", "validation"], default="test")
schema = json.loads((ROOT / "data/checkout_features.json").read_text())
features = sum(schema.values(), [])
metadata = ["order_id", "purchased_at", "label_observed_at", "split", "late"]
assert len(features) == len(set(features)) and not set(metadata).intersection(features)
assert not np.isinf(frame[schema["log_numeric"] + schema["numeric"]].to_numpy()).any()
assert frame.loc[frame.split.eq("train"), "purchased_at"].max() < val_start < test_start
summary = frame.groupby("split", sort=False).agg(orders=("order_id", "size"),
                                                  first_purchase=("purchased_at", "min"),
                                                  last_purchase=("purchased_at", "max"))
display(summary)
summary.to_csv(TABLES / "purchase_splits.csv")
frame[metadata + features].to_parquet(PROCESSED / "orders.parquet", index=False)
print(f"Saved {len(frame):,} orders with {len(features)} checkout predictors.")

,orders,first_purchase,last_purchase
split,,,
train,67379,2016-09-15 12:16:38,2018-04-14 23:28:11
validation,14606,2018-04-15 00:08:41,2018-06-20 23:39:24
test,14485,2018-06-21 00:22:02,2018-08-29 15:00:37


Saved 96,470 orders with 19 checkout predictors.


The pinned release yields 96,470 eligible orders from 99,441 purchases. A calendar-day deadline labels 6,534 orders late. Moving the deadline to midnight would add 1,292 deliveries that arrived during the promised day. Of the retained orders, 1,275 span several sellers and 481 lack a complete route distance.